# Inspect Disconnect Fees DataFrame

This notebook inspects the DataFrame structure for the Disconnect Fees sheet to understand how duplicate validation should work for fee sheets that have separate columns for each month instead of a single "Month" column.

## Import Required Libraries

In [ ]:
import polars as pl

import nwec.utils.excel
from nwec.constants import MONTHS, RAW_UTILITY_DATA
from nwec.utils.cleaning import clean_utility_data, validate_data

MASTER_SPREADSHEET = RAW_UTILITY_DATA / "IOU 200281 Data.xlsx"

## Load the Raw Disconnect Fees Data

First, let's load the raw data from the Excel sheet to see the original structure.

In [ ]:
sheet_name = "Disconnect Fees"
sheet_index = nwec.utils.excel.get_sheet_index_from_name(MASTER_SPREADSHEET, sheet_name)
df_raw = pl.read_excel(MASTER_SPREADSHEET, sheet_id=sheet_index, has_header=False)

print(f"Raw DataFrame shape: {df_raw.shape}")
print(f"\nRaw DataFrame columns: {df_raw.columns}")
print("\nFirst few rows:")
df_raw.head(10)

## Process the Header

The data processing pipeline promotes the second row as the header.

In [ ]:
# Get rid of the first row and promote the second to be the DataFrame header
headers = df_raw.slice(1, 1).row(0)
df = df_raw.slice(2).rename(dict(zip(df_raw.columns, [str(col) for col in headers], strict=True)))

print(f"DataFrame with proper headers shape: {df.shape}")
print(f"\nColumns: {df.columns}")
print("\nFirst few rows:")
df.head(10)

## Identify Month Columns and Index Columns

Fee sheets have separate columns for each month instead of a single "Month" column.

In [ ]:
print(f"MONTHS constant: {MONTHS}")

# Identify which columns are month columns vs index columns
month_columns = [col for col in df.columns if col in MONTHS]
index_cols = [col for col in df.columns if col not in MONTHS]

print(f"\nMonth columns found: {month_columns}")
print(f"\nIndex (key) columns: {index_cols}")

## Unpivot the Data (Melt Month Columns)

The standard pipeline unpivots the month columns into a single "Month" column.

In [ ]:
# Unpivot the month columns
value_column_name = "Disconnection Fee Amount"
df_unpivoted = df.unpivot(on=month_columns, index=index_cols, variable_name="Month", value_name=value_column_name)

# Convert month names to integers (1-12)
df_unpivoted = df_unpivoted.with_columns(pl.col("Month").str.to_datetime("%B").dt.month().alias("Month"))

print(f"Unpivoted DataFrame shape: {df_unpivoted.shape}")
print(f"\nColumns: {df_unpivoted.columns}")
print("\nFirst 20 rows:")
df_unpivoted.head(20)

## Apply Standard Filtering and Cleaning

The pipeline filters for residential customers and applies data cleaning.

In [ ]:
# Filter for residential customers and drop the Customer Class column
if "Customer Class" in df_unpivoted.columns:
    df_filtered = df_unpivoted.filter(pl.col("Customer Class").str.contains(r"(?i)res")).drop("Customer Class")
else:
    df_filtered = df_unpivoted

print(f"Filtered DataFrame shape: {df_filtered.shape}")
print(f"\nColumns: {df_filtered.columns}")
print("\nFirst 20 rows:")
df_filtered.head(20)

## Apply clean_utility_data()

This function cleans and converts data types.

In [ ]:
df_cleaned = clean_utility_data(df_filtered, value_column_name=value_column_name)

print(f"Cleaned DataFrame shape: {df_cleaned.shape}")
print(f"\nColumns: {df_cleaned.columns}")
print("\nData types:")
print(df_cleaned.dtypes)
print("\nFirst 20 rows:")
df_cleaned.head(20)

## Check for Duplicates

This is what _validate_duplicates() checks - are there duplicate rows based on key columns?

In [ ]:
# Build list of key columns that exist in the dataframe
key_columns = [
    col for col in ["Utility", "Year", "Month", "Customer Class", "Zip Code", "Vintage"] if col in df_cleaned.columns
]

print(f"Key columns for duplicate checking: {key_columns}")

# Check for duplicates based on key columns
duplicate_check = df_cleaned.select(key_columns).group_by(key_columns).agg(pl.len().alias("count"))
duplicates = duplicate_check.filter(pl.col("count") > 1)

print(f"\nNumber of duplicate key combinations: {len(duplicates)}")
if len(duplicates) > 0:
    print("\nSample duplicates:")
    print(duplicates.head(10))

## Show the Full DataFrame That Would Be Passed to _validate_duplicates()

This is the final DataFrame after all cleaning that gets passed to the validation functions.

In [ ]:
print("This is the DataFrame that gets passed to validate_data() and _validate_duplicates():")
print(f"\nShape: {df_cleaned.shape}")
print(f"\nColumns: {df_cleaned.columns}")
print(f"\nData types: {df_cleaned.dtypes}")
print("\nSummary statistics:")
print(df_cleaned.describe())
print("\nAll data:")
df_cleaned